<a href="https://colab.research.google.com/github/withfablue/PyTorch/blob/main/Chapter3_Data_Pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

실습: 신경망 학습 루프 구현하기

In [ ]:
#1. 데이터셋과 데이터로더 구성
import torch
from torch.utils.data import Dataset, DataLoader

# 간단한 데이터셋 정의
class SimpleDataset(Dataset):
  def __init__(self):
    torch.manual_seed(0)
    self.X = torch.randn(200, 2)
    self.y = (self.X[:,0] + self.X[:,1] > 0).long()

  def __len__(self):
    return len(self.X)

  def __getitem__(self, idx):
    return self.X[idx], self.y[idx]


dataset = SimpleDataset()
loader = DataLoader(dataset, batch_size=16, shuffle=True)

In [ ]:
#1. 데이터셋과 데이터로더 구성
import torch
from torch.utils.data import Dataset, DataLoader

# 간단한 데이터셋 정의
class SimpleDataset(Dataset):
  def __init__(self):
    torch.manual_seed(0)
    self.X = torch.randn(200, 2)
    self.y = (self.X[:,0] + self.X[:,1] > 0).long()

  def __len__(self):
    return len(self.X)

  def __getitem__(self, idx):
    return self.X[idx], self.y[idx]

dataset = SimpleDataset()
loader = DataLoader(dataset, batch_size=16, shuffle=True)

In [ ]:
# 2. 모델, 손실함수, 최적화 알고리즘 정의
# Binary Classfication using MLP with 1 hidden layer(hidden size=4)

import torch
import torch.nn as nn
import torch.optim as optim

model = nn.Sequential(
    nn.Linear(2, 4),
    nn.ReLU(),
    nn.Linear(4, 1),
    nn.Sigmoid()
)

criterion = nn.BCELoss()
optimizer = optim.SGD(model.parameters(), lr=0.1)

In [ ]:
# 2. 모델, 손실함수, 최적화 알고리즘 정의

import torch.nn as nn
import torch.optim as optim

model = nn.Sequential(
    nn.Linear(2, 4)
    nn.ReLU(),
    nn.Linear(4, 1),
    nn.Sigmoid()
)

criterion = nn.BCELoss()
optimizer = optim.SGD(model.parameters(), lr=0.1)

In [ ]:
# 3. 학습 루프 구현
# forward prop -> loss calculation -> backprop & parameter update

num_epochs = 10

for epoch in range(num_epochs):
  epoch_loss = 0.0

  for X_batch, y_batch in loader:
    optimizer.zero_grad()        # 기울기 초기화
    outputs = model(X_batch).squeeze() # 순전파 수행
    loss = criterion(outputs, y_batch.float())  # 손실 계산
    loss.backward()              # 역전파 수행 (기울기 계산)
    optimizer.step()             # 파라미터 업데이트

    epoch_loss += loss.item()

  print(f"Epoch {epoch+1}: Loss = {epoch_loss:.4f}")



Epoch 1: Loss = 8.9390
Epoch 2: Loss = 8.4666
Epoch 3: Loss = 7.9390
Epoch 4: Loss = 7.2274
Epoch 5: Loss = 6.5833
Epoch 6: Loss = 6.0083
Epoch 7: Loss = 5.5505
Epoch 8: Loss = 5.1448
Epoch 9: Loss = 4.7004
Epoch 10: Loss = 4.3583


In [ ]:
# 3. 학습 루프 구현

num_epochs = 10

for epoch in range(num_epochs):
  epoch_loss = 0.0

  for X_batch, y_batch in loader:
    optimizer.zero_grad()
    outputs = model(X_batch).squeeze()
    loss = criterion(outputs, y_batch.float())
    loss.backward()
    optimizer.step()

    epoch_loss += loss.item()

  print(f"Epoch {epoch+1}: Loss = {epoch_loss:.4f}")

Epoch 1: Loss = 4.1847
Epoch 2: Loss = 3.8376
Epoch 3: Loss = 3.5648
Epoch 4: Loss = 3.4421
Epoch 5: Loss = 3.1875
Epoch 6: Loss = 3.0772
Epoch 7: Loss = 2.9081
Epoch 8: Loss = 2.7362
Epoch 9: Loss = 2.6738
Epoch 10: Loss = 2.5103


In [ ]:
# 4. 모델 평가
with torch.no_grad():
  X_all = dataset.X
  y_all = dataset.y

  preds = (model(X_all).squeeze() > 0.5).long()
  accuracy = (preds == y_all).float().mean().item()

print("Training Accuracy:", accuracy)


Training Accuracy: 0.9800000190734863


In [ ]:
# 4. 모델 평가
with torch.no_grad():
  X_all = dataset.X
  y_all = dataset.y

  preds = (model(X_all).squeeze() > 0.5).long()
  accuracy = (preds==y_all).float().mean().item()

print("Training Accuracy:", accuracy)

Training Accuracy: 0.9800000190734863


In [ ]:
# 실습 문제 1. NormalizedDataset 작성

import torch
from torch.utils.data import Dataset, DataLoader

class NormalizedDataset(Dataset):
  def __init__(self):
    torch.manual_seed(0)
    self.X = torch.randn(200,2)
    self.y = (self.X[:,0] + self.X[:,1] > 0).long()
    self.mean = self.X.mean(dim=0)
    self.std = self.X.std(dim=0)
    self.normalized_X = (self.X - self.mean) / self.std

  def __len__(self):
    return len(self.X)

  def __getitem__(self, idx):
    return self.normalized_X[idx], self.y[idx]


normset = NormalizedDataset()
print(normset.mean, normset.std)
print(normset.normalized_X.mean(dim=0), normset.normalized_X.std(dim=0))

tensor([0.0883, 0.0366]) tensor([0.8928, 1.0637])
tensor([-9.5367e-09,  9.5367e-09]) tensor([1.0000, 1.0000])


In [ ]:
# 실습 문제 2. 데이터로더의 배치 구성 확인

dataset = SimpleDataset()
loader = DataLoader(dataset, batch_size=16, shuffle=True)

for X_batch, y_batch in loader:
  print(X_batch.shape, y_batch.shape)

print(len(X_batch))

# 마지막 샘플 개수가 배치 크기와 다를 수 있다.

torch.Size([16, 2]) torch.Size([16])
torch.Size([16, 2]) torch.Size([16])
torch.Size([16, 2]) torch.Size([16])
torch.Size([16, 2]) torch.Size([16])
torch.Size([16, 2]) torch.Size([16])
torch.Size([16, 2]) torch.Size([16])
torch.Size([16, 2]) torch.Size([16])
torch.Size([16, 2]) torch.Size([16])
torch.Size([16, 2]) torch.Size([16])
torch.Size([16, 2]) torch.Size([16])
torch.Size([16, 2]) torch.Size([16])
torch.Size([16, 2]) torch.Size([16])
torch.Size([8, 2]) torch.Size([8])
8


In [ ]:
# 실습 문제 3. batch_size 변화에 따른 업데이트 횟수 비교
dataset = SimpleDataset()
loader8 = DataLoader(dataset, batch_size=8, shuffle=True)
loader16 = DataLoader(dataset, batch_size=16, shuffle=True)
loader32 = DataLoader(dataset, batch_size=32, shuffle=True)

count8 = 0
for X_batch, y_batch in loader8:
  optimizer.zero_grad()
  outputs = model(X_batch).squeeze()
  loss = criterion(outputs, y_batch.float())
  loss.backward()
  optimizer.step()
  count8 += 1

count16 = 0
for X_batch, y_batch in loader16:
  optimizer.zero_grad()
  outputs = model(X_batch).squeeze()
  loss = criterion(outputs, y_batch.float())
  loss.backward()
  optimizer.step()
  count16 += 1

count32 = 0
for X_batch, y_batch in loader32:
  optimizer.zero_grad()
  outputs = model(X_batch).squeeze()
  loss = criterion(outputs, y_batch.float())
  loss.backward()
  optimizer.step()
  count32 += 1

print(count8, count16, count32)

# 배치 크기가 클수록 업데이트 횟수가 줄어드는 이유
배치 크기가 클수록 한 epoch에 해당하는 루프가 줄어들기 때문에 (전체 데이터셋에 걸쳐 한 번씩 업데이트하기 위한 루프의 수가 줄어들기 때문에) 업데이트 횟수도 적다.

25 13 7


In [ ]:
# 실습 문제 4. shuffle 옵션에 따른 배치 조합 비교

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

class IndexDataset(Dataset):
  def __init__(self):
    torch.manual_seed(0)
    self.X = torch.randn(200, 2)
    self.y = (self.X[:,0]+self.X[:,1]>0).long()

  def __len__(self):
    return len(self.X)

  def __getitem__(self, idx):
    return idx, idx


dataset = IndexDataset()
loader1 = DataLoader(dataset, batch_size=8, shuffle=False)
loader2 = DataLoader(dataset, batch_size=8, shuffle=True)

for step, (X, y) in enumerate(loader1):
  print(X)
  if step == 1:
    break

for step, (X, y) in enumerate(loader2):
  print(X)
  if step == 1:
    break

# shuffle=True인 경우 데이터 순서가 무작위로 섞여 배치가 구성됨.

tensor([0, 1, 2, 3, 4, 5, 6, 7])
tensor([ 8,  9, 10, 11, 12, 13, 14, 15])
tensor([ 23,  88,  21, 142, 149, 156,  66,  31])
tensor([ 99,  17,  38, 173, 185,  81,   0, 100])


In [ ]:
# 실습 문제 5: 지연 로딩 데이터셋 구현

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

class OnDemandDataset(Dataset):
  def __init__(self, N):
    self.N = N

  def __len__(self):
    return self.N

  def __getitem__(self, idx):
    torch.manual_seed(idx)
    x = torch.randn(2)
    y = (x[0] + x[1] > 0).long()
    return x, y

dataset = OnDemandDataset(100)
print(dataset[100])
print(dataset[100])

(tensor([ 0.3607, -0.2859]), tensor(1))
(tensor([ 0.3607, -0.2859]), tensor(1))


In [ ]:
# 실습 문제 6. 전처리 캐싱 구현
# 사전에 전처리 결과를 저장해 재사용하는 캐시 전략 구현

import torch
from torch.utils.data import Dataset, DataLoader

class CacheDataset(Dataset):
  def __init__(self):
    torch.manual_seed(0)
    self.X = torch.randn(200,2)
    self.y = (self.X[:,0] + self.X[:,1] > 0).long()
    self.cache = dict()
    self.count = 0

  def __len__(self):
    return len(self.X)

  def __getitem__(self, idx):
    if idx in self.cache:
      return self.cache[idx], self.y[idx]
    else:
      self.cache[idx] = preprocess(self.X[idx])
      self.count += 1
      return self.cache[idx], self.y[idx]

def preprocess(x):
  return torch.abs(torch.sin(x)**2 + x)


dataset = CacheDataset()
print(dataset[99])
print(dataset.count)
print(dataset[99])
print(dataset.count)
print(dataset[99])
print(dataset.count)
print(dataset[9])
print(dataset.count)
print(dataset[9])
print(dataset.count)



(tensor([0.2069, 1.7492]), tensor(1))
1
(tensor([0.2069, 1.7492]), tensor(1))
1
(tensor([0.2069, 1.7492]), tensor(1))
1
(tensor([0.8548, 1.3016]), tensor(1))
2
(tensor([0.8548, 1.3016]), tensor(1))
2
